## Tahap 0: Import dan Lokasi Penyimpanan

Hasil scraping disimpan di `Dataset/tugas/`. Berkas `leetcode_mentah.jsonl`
berfungsi ganda, sebagai hasil sekaligus checkpoint.

In [1]:
import json
import time
from pathlib import Path

import requests
import pandas as pd
from bs4 import BeautifulSoup


def cari_root(mulai: Path) -> Path:
    """Cari root project, yaitu folder yang memuat subfolder 'Praktikum'."""
    for p in (mulai, *mulai.parents):
        if (p / "Praktikum").is_dir():
            return p
    return mulai


ROOT = cari_root(Path.cwd().resolve())
OUT_DIR = ROOT / "Dataset" / "tugas"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FILE_DAFTAR = OUT_DIR / "daftar_soal.json"        # metadata semua soal
FILE_MENTAH = OUT_DIR / "leetcode_mentah.jsonl"   # hasil sekaligus checkpoint
FILE_CSV = OUT_DIR / "leetcode_soal.csv"
FILE_XLSX = OUT_DIR / "leetcode_soal.xlsx"

print(f"Root project : {ROOT}")
print(f"Folder output: {OUT_DIR}")

Root project : /Users/alif/PycharmProjects/Text-Mining
Folder output: /Users/alif/PycharmProjects/Text-Mining/Dataset/tugas


## Tahap 1: Query GraphQL

Dua query yang sama persis dengan notebook praktikum. Bedanya query daftar
sekarang ikut mengambil `questionFrontendId` dan `isPaidOnly`, supaya nomor soal
tercatat dan soal premium bisa dilewati sejak awal.

In [2]:
url = "https://leetcode.com/graphql"

query_problem_list = """
query problemsetQuestionList($limit: Int, $skip: Int) {
  problemsetQuestionList: questionList(
    categorySlug: ""
    limit: $limit
    skip: $skip
    filters: {}
  ) {
    total: totalNum
    questions: data {
      questionFrontendId
      title
      titleSlug
      difficulty
      isPaidOnly
      topicTags {
        name
      }
    }
  }
}
"""

query_problem_content = """
query questionData($titleSlug: String!) {
  question(titleSlug: $titleSlug) {
    content
  }
}
"""


def kirim_query(query, variables, max_retry=3):
    """Kirim request GraphQL dengan retry, supaya proses panjang tidak gagal total."""
    for percobaan in range(1, max_retry + 1):
        try:
            response = requests.post(url, json={'query': query, 'variables': variables}, timeout=30)
            response.raise_for_status()
            return response.json()['data']
        except Exception as e:
            if percobaan == max_retry:
                raise
            print(f"   (gagal, mencoba lagi {percobaan}/{max_retry}: {e})")
            time.sleep(2 * percobaan)

## Tahap 2: Mengambil Daftar Seluruh Soal

Endpoint daftar sudah menyertakan `topicTags`, jadi seluruh label multilabel bisa
didapat hanya dengan sekitar 40 request. Yang mahal nanti adalah deskripsinya,
karena itu butuh satu request per soal.

API membatasi 100 soal per request, jadi diambil bertahap lewat `skip`. Urutan
antar halaman kadang tidak stabil dan bisa menghasilkan duplikat, jadi setiap slug
dicek dulu sebelum disimpan.

In [3]:
if FILE_DAFTAR.exists():
    daftar_semua = json.loads(FILE_DAFTAR.read_text())
    print(f"Memakai daftar tersimpan: {len(daftar_semua)} soal")
else:
    daftar_semua, sudah_ada, skip = [], set(), 0
    while True:
        blok = kirim_query(query_problem_list, {"limit": 100, "skip": skip})['problemsetQuestionList']
        batch = blok['questions']
        if not batch:
            break
        for soal in batch:
            if soal['titleSlug'] not in sudah_ada:       # buang duplikat antar halaman
                sudah_ada.add(soal['titleSlug'])
                daftar_semua.append(soal)
        skip += len(batch)
        print(f"  {len(daftar_semua)} soal terdaftar (dari {blok['total']})")
        if skip >= blok['total']:
            break
        time.sleep(0.4)

    FILE_DAFTAR.write_text(json.dumps(daftar_semua))
    print(f"\nDaftar disimpan ke {FILE_DAFTAR.name}")

daftar_gratis = [s for s in daftar_semua if not s['isPaidOnly']]
print(f"\nTotal terdaftar : {len(daftar_semua)}")
print(f"Gratis          : {len(daftar_gratis)}")
print(f"Premium dilewati: {len(daftar_semua) - len(daftar_gratis)}")

Memakai daftar tersimpan: 4069 soal

Total terdaftar : 4069
Gratis          : 3284
Premium dilewati: 785


## Tahap 3: Fungsi Scraping Deskripsi

Ini kode yang sama dengan notebook praktikum, dengan dua perubahan.

Pengecekan `soup.find('img')` dihapus, jadi soal bergambar ikut diambil. Soal
premium tetap dilewati karena `content` yang dikirim server bernilai `None`.

Checkpoint ditulis dalam format JSONL, satu soal satu baris. Tiap soal yang
selesai langsung ditambahkan ke berkas. Saat dijalankan ulang, slug yang sudah ada
di berkas dilewati, jadi proses lanjut dari posisi terakhir tanpa mengulang.

In [4]:
def scrape_deskripsi(daftar, file_checkpoint, jeda=0.5, lapor_tiap=100):
    """Ambil deskripsi tiap soal. Bisa dijalankan ulang, lanjut dari checkpoint."""
    sudah = set()
    if file_checkpoint.exists():
        with open(file_checkpoint, encoding="utf-8") as f:
            for baris in f:
                try:
                    sudah.add(json.loads(baris)['titleSlug'])
                except Exception:
                    continue   # baris rusak akibat proses terputus, abaikan

    sisa = [s for s in daftar if s['titleSlug'] not in sudah]
    print(f"Sudah terambil : {len(sudah)}")
    print(f"Sisa diproses  : {len(sisa)}")
    if not sisa:
        print("Semua soal sudah terambil.")
        return

    perkiraan = len(sisa) * (jeda + 0.25) / 60
    print(f"Perkiraan waktu: {perkiraan:.0f} menit\n")

    mulai = time.time()
    gagal = 0
    with open(file_checkpoint, "a", encoding="utf-8") as f:
        for i, soal in enumerate(sisa, 1):
            try:
                content_html = kirim_query(
                    query_problem_content, {"titleSlug": soal['titleSlug']}
                )['question']['content']
            except Exception as e:
                gagal += 1
                print(f"   gagal pada {soal['titleSlug']}: {e}")
                continue

            if content_html is None:      # soal premium, isinya tidak dikirim
                continue

            soup = BeautifulSoup(content_html, 'html.parser')
            f.write(json.dumps({
                'questionFrontendId': soal['questionFrontendId'],
                'titleSlug': soal['titleSlug'],
                'title': soal['title'],
                'difficulty': soal['difficulty'],
                'topicTags': [t['name'] for t in soal['topicTags']],
                'description': soup.get_text(separator=" ", strip=True),
                'ada_gambar': soup.find('img') is not None,
            }, ensure_ascii=False) + "\n")

            if i % 25 == 0:
                f.flush()      # pastikan tertulis ke disk, bukan menumpuk di memori

            if i % lapor_tiap == 0:
                lewat = time.time() - mulai
                sisa_menit = (len(sisa) - i) * (lewat / i) / 60
                print(f"  {i}/{len(sisa)} selesai | {lewat/60:.1f} menit berjalan "
                      f"| perkiraan sisa {sisa_menit:.0f} menit")

            time.sleep(jeda)

    print(f"\nSelesai dalam {(time.time() - mulai)/60:.1f} menit, {gagal} soal gagal diambil.")

## Tahap 4: Menjalankan Scraping

Sel ini yang memakan waktu paling lama, sekitar 25 menit untuk 3.284 soal.

Kalau prosesnya terputus karena apa pun, jalankan ulang sel ini saja. Soal yang
sudah terambil akan dilewati otomatis.

Nilai `jeda` adalah waktu tunggu antar request dalam detik. Jangan diturunkan
terlalu jauh di bawah 0,5 supaya tidak dianggap serangan oleh server LeetCode.

In [5]:
scrape_deskripsi(daftar_gratis, FILE_MENTAH, jeda=0.5)

Sudah terambil : 3284
Sisa diproses  : 0
Semua soal sudah terambil.


## Tahap 5: Menyusun Tabel Hasil

Berkas JSONL dibaca ulang jadi DataFrame. Kolom `Topic Tags` disimpan sebagai teks
dipisah koma supaya tetap terbaca di CSV maupun Excel, dan nanti tinggal dipecah
lagi saat tahap pelabelan multilabel.

In [6]:
baris = []
with open(FILE_MENTAH, encoding="utf-8") as f:
    for teks in f:
        try:
            baris.append(json.loads(teks))
        except Exception:
            continue

df = pd.DataFrame(baris).drop_duplicates(subset='titleSlug').reset_index(drop=True)
df = df.rename(columns={
    'questionFrontendId': 'Problem ID',
    'title': 'Problem Name',
    'titleSlug': 'Slug',
    'difficulty': 'Difficulty',
    'description': 'Description',
    'ada_gambar': 'Ada Gambar',
})
df['Topic Tags'] = df['topicTags'].apply(lambda t: ", ".join(t))
df['Jumlah Tag'] = df['topicTags'].apply(len)
# 'topicTags' (bentuk list) sengaja ikut dibawa karena dipakai Tahap 6.
# Kolom itu baru dibuang saat menyimpan ke CSV di Tahap 7.
df = df[['Problem ID', 'Problem Name', 'Slug', 'Difficulty',
         'Topic Tags', 'Jumlah Tag', 'Ada Gambar', 'Description', 'topicTags']]

print(f"Jumlah soal terkumpul: {len(df)}")
print(f"Kolom                : {list(df.columns)}\n")
display(df.head())

Jumlah soal terkumpul: 3284
Kolom                : ['Problem ID', 'Problem Name', 'Slug', 'Difficulty', 'Topic Tags', 'Jumlah Tag', 'Ada Gambar', 'Description', 'topicTags']



,Problem ID,Problem Name,Slug,Difficulty,Topic Tags,Jumlah Tag,Ada Gambar,Description,topicTags
0,1,Two Sum,two-sum,Easy,"Array, Hash Table",2,False,You are given an array of integers nums and an...,"[Array, Hash Table]"
1,2,Add Two Numbers,add-two-numbers,Medium,"Linked List, Math, Recursion",3,True,You are given two non-empty linked lists repre...,"[Linked List, Math, Recursion]"
2,3,Longest Substring Without Repeating Characters,longest-substring-without-repeating-characters,Medium,"Hash Table, String, Sliding Window",3,False,"Given a string s , find the length of the long...","[Hash Table, String, Sliding Window]"
3,4,Median of Two Sorted Arrays,median-of-two-sorted-arrays,Hard,"Array, Binary Search, Divide and Conquer",3,False,Given two sorted arrays nums1 and nums2 of siz...,"[Array, Binary Search, Divide and Conquer]"
4,5,Longest Palindromic Substring,longest-palindromic-substring,Medium,"Two Pointers, String, Dynamic Programming, Man...",4,False,"Given a string s , return the longest palindro...","[Two Pointers, String, Dynamic Programming, Ma..."


## Tahap 6: Memeriksa Hasil

Beberapa angka untuk memastikan datanya layak sebelum masuk tahap pelabelan.

Yang perlu diperhatikan: rata rata jumlah tag per soal harus di atas satu, karena
itulah yang membuat data ini multilabel. Kalau hampir semua soal cuma punya satu
tag, tugas multilabelnya jadi tidak ada artinya.

In [7]:
print("Distribusi Difficulty:")
print(df['Difficulty'].value_counts().to_string())

print(f"\nSoal yang mengandung gambar: {df['Ada Gambar'].sum()} "
      f"({100 * df['Ada Gambar'].mean():.1f} persen, tetap diambil)")

print(f"\nJumlah tag per soal:")
print(f"  rata rata : {df['Jumlah Tag'].mean():.2f}")
print(f"  soal tanpa tag : {(df['Jumlah Tag'] == 0).sum()}")
print(f"  soal 1 tag     : {(df['Jumlah Tag'] == 1).sum()}")
print(f"  soal 2 tag ke atas : {(df['Jumlah Tag'] >= 2).sum()}")
print(f"  maksimum  : {df['Jumlah Tag'].max()}")

import collections
semua_tag = collections.Counter(t for daftar in df['topicTags'] for t in daftar)
print(f"\nTag unik: {len(semua_tag)}")
print("15 tag terbanyak:")
for tag, n in semua_tag.most_common(15):
    print(f"  {tag:<28} {n:>5}")

Distribusi Difficulty:
Difficulty
Medium    1660
Easy       839
Hard       785

Soal yang mengandung gambar: 738 (22.5 persen, tetap diambil)

Jumlah tag per soal:
  rata rata : 3.30
  soal tanpa tag : 64
  soal 1 tag     : 317
  soal 2 tag ke atas : 2903
  maksimum  : 15

Tag unik: 173
15 tag terbanyak:
  Array                         1947
  String                         772
  Hash Table                     715
  Math                           619
  Dynamic Programming            600
  Sorting                        465
  Greedy                         434
  Binary Search                  306
  Depth-First Search             265
  Bit Manipulation               264
  Prefix Sum                     249
  Matrix                         240
  Two Pointers                   221
  Breadth-First Search           204
  Tree                           201


## Tahap 7: Menyimpan Dataset

Disimpan dalam dua format. CSV dipakai untuk pemrosesan lanjutan karena ringan dan
cepat dibaca, XLSX untuk dibuka manual kalau perlu diperiksa mata.

In [8]:
simpan = df.drop(columns=['topicTags'])
simpan.to_csv(FILE_CSV, index=False)
simpan.to_excel(FILE_XLSX, index=False)

print(f"Tersimpan: {FILE_CSV}")
print(f"Tersimpan: {FILE_XLSX}\n")
for berkas in sorted(OUT_DIR.iterdir()):
    print(f"  {berkas.name:<26} {berkas.stat().st_size / 1024 / 1024:>7.2f} MB")

cek = pd.read_csv(FILE_CSV)
print(f"\nVerifikasi baca ulang: {cek.shape[0]} baris, {cek.shape[1]} kolom")

Tersimpan: /Users/alif/PycharmProjects/Text-Mining/Dataset/tugas/leetcode_soal.csv
Tersimpan: /Users/alif/PycharmProjects/Text-Mining/Dataset/tugas/leetcode_soal.xlsx

  daftar_soal.json              1.01 MB
  hasil                         0.00 MB
  leetcode_berlabel.csv         0.32 MB
  leetcode_mentah.jsonl         4.65 MB
  leetcode_soal.csv             4.26 MB
  leetcode_soal.xlsx            1.23 MB

Verifikasi baca ulang: 3284 baris, 8 kolom


## Selesai

Data mentah siap. Langkah berikutnya sesuai rencana yang sudah dibahas:

1. Saring tag yang punya minimal 50 soal, menyisakan sekitar 34 label
2. Ubah kolom `Topic Tags` jadi matriks biner dengan `MultiLabelBinarizer`
3. Validasi manual 100 soal sebagai bukti anotasi sudah diperiksa
4. Vektorisasi teks, lalu klasifikasi multilabel dengan Binary Relevance
   dibandingkan Classifier Chains